# 04 | Variable respuesta avalúo

> **Cuaderno de reproducción (posterior).** Generado el 2026-10-23 como parte de la entrega docente. El repositorio original no contenía cuadernos versionados; estos reconstruyen los análisis a partir de los scripts y módulos existentes.

**Tipo de ejecución: CARGA DE DATOS — lee los CSV crudos; NO entrena modelos.**

In [1]:
import os
from pathlib import Path

def _raiz_repositorio() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data' / 'raw' / 'interseccion_terrenos_zhg_r1_yacopi.csv').exists():
            return p
    raise FileNotFoundError('No se encontro la raiz del repositorio (busca data/raw)')

REPO = _raiz_repositorio()
print('Raiz del repositorio resuelta (carpeta):', REPO.name, '| data/raw encontrado')

Raiz del repositorio resuelta (carpeta): yacopi-project | data/raw encontrado


## VALOR_TERRENO_Ha = VALOR_AVALUO × 10000 / AREA_TERRENO_M2

Nueva variable comunicada por el profesor. El avalúo es catastral TOTAL; no hay campo de valor exclusivo de terreno.

In [2]:
import sys
sys.path.insert(0, str(REPO / 'ml' / 'src'))
from yacopi_ml.valor.dataset import cargar_intersecciones
df = cargar_intersecciones()
df['valor_terreno_ha'] = df['valor_avaluo'] * 10000 / df['area_terreno_m2']
print('valores unicos:', df['valor_terreno_ha'].nunique())
print(df['valor_terreno_ha'].describe().round(0).to_string())

valores unicos: 5654
count    9.447000e+03
mean     8.153664e+06
std      5.755657e+07
min      9.200000e+04
25%      2.934600e+05
50%      4.820740e+05
75%      9.643480e+05
max      1.481689e+09


In [3]:
import numpy as np
print('Igualdad exacta con zonal x area: %: ',
      np.isclose(df['valor_avaluo'],
                 df['area_terreno_m2']*df['valor_hectarea_agropecuario']/10000,
                 rtol=1e-6).mean()*100)
print('corr(avaluo, zonal x area) =',
      df[['valor_avaluo','valor_hectarea_agropecuario']].corr().iloc[0,1].round(3))

Igualdad exacta con zonal x area: %:  0.0
corr(avaluo, zonal x area) = 0.197


## Lectura

La variable NO es copia del valor zonal (0 % de igualdad exacta; 5.654 valores únicos), pero está fuertemente asociada a él (corr 0,73). No es valor comercial.